# MeasureX — Real GPU Validation & Warehouse Box Benchmark (M6)

Runs the **real** camera-only pipeline (M0–M5) on a physical carton and benchmarks the predicted L/W/H against your tape-measured ground truth. **No mocked inference.**

- Ordinary RGB phone photo/video — **no mat, no ArUco, no reference object, no LiDAR/ARCore/ARKit.**
- Depth: **Depth Pro** — `PRODUCTION_ALLOWED = false` (Apple-ASCL, research/benchmark only).
- Segmentation: **SAM 2** (Apache-2.0). Detector: model-free central-candidate.
- Depth Anything V2 Small (commercial-path candidate) is **deferred** this milestone.
- First: **Runtime → Change runtime type → GPU**. This does **not** touch the Flutter app.

See `MODELS.md` for licenses and `docs/REAL_GPU_VALIDATION_REPORT.md` for the report to fill in.

## 1. Verify GPU (stops clearly if no CUDA)

In [ ]:
!nvidia-smi

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No CUDA GPU — Runtime > Change runtime type > GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
print('device count:', torch.cuda.device_count())

## 2. Clone the repo (branch `main`)

In [ ]:
%cd /content
!rm -rf MeasureX
!git clone https://github.com/Shaikhabdulaleem/MeasureX.git
%cd MeasureX/services/vision-poc
!git checkout main && git log --oneline -1

## 3. Install the vision POC + model stack

`depth` = torch + Depth Pro (Apple git); `seg` = torch + SAM 2 (Meta git). Colab already has a CUDA torch, so this mainly pulls the two model packages. A few minutes.

Note: **non-editable** install (no `-e`) — an editable install is not reliably importable by the `!python …` subprocesses used below in Colab. The verify line fails loudly if anything didn't install.

In [ ]:
!pip install -q ".[depth,seg]"
import measurex_vision
print('measurex_vision', measurex_vision.__version__, 'installed OK')

## 4. Download model weights + set config (fails clearly if missing)

In [ ]:
import os
os.makedirs('weights', exist_ok=True)
# Depth Pro (Apple-ASCL, research/benchmark only) -> weights/depth_pro.pt
!python scripts/download_models.py
# SAM 2.1 large (Apache-2.0)
!wget -q -O weights/sam2.1_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

root = '/content/MeasureX/services/vision-poc'
os.environ['MEASUREX_DEPTH_CHECKPOINT'] = f'{root}/weights/depth_pro.pt'
os.environ['MEASUREX_SAM2_CHECKPOINT']  = f'{root}/weights/sam2.1_hiera_large.pt'
os.environ['MEASUREX_SAM2_CONFIG']      = 'configs/sam2.1/sam2.1_hiera_l.yaml'
for p in [os.environ['MEASUREX_DEPTH_CHECKPOINT'], os.environ['MEASUREX_SAM2_CHECKPOINT']]:
    assert os.path.exists(p) and os.path.getsize(p) > 1_000_000, f'weight missing/too small: {p}'
print('weights OK')

## 5. Enter the real box ground truth (tape-measured, mm)

Measure the carton with a tape: **Length** = longest horizontal, **Width** = shorter horizontal, **Height** = vertical. Edit the three numbers, then run — this writes `ground_truth.json`.

In [ ]:
import json, os
OBJECT_ID = 'BOX_001'
L, W, H = 600, 400, 350   # <-- EDIT: your real measured millimetres (length >= width)
assert L >= W, 'length must be >= width'
os.makedirs(f'data/benchmark/{OBJECT_ID}', exist_ok=True)
gt = {'object_id': OBJECT_ID, 'category': 'carton', 'length_mm': L, 'width_mm': W, 'height_mm': H}
json.dump(gt, open(f'data/benchmark/{OBJECT_ID}/ground_truth.json', 'w'), indent=2)
print(gt)

## 6. Capture & upload your box photo and video

**Photos (3–5, JPG):** whole carton in frame, some floor/ground visible around the base, slight angle (see top + two sides), decent light, minimal blur, nothing touching the box, no mat/ruler.

**Video (one, MP4, 3–5 s):** move slowly a little around the box (do not circle fully), whole box in frame throughout.

Run the cell and pick **one JPG** and **one MP4** (you can re-run for more photos).

In [ ]:
from google.colab import files
up = files.upload()           # choose your box PHOTO (jpg)
img = next(iter(up))
upv = files.upload()          # choose your box VIDEO (mp4)
vid = next(iter(upv))
print('image:', img, '| video:', vid)

## 7. Run image inference (real pipeline)

detection → segmentation → depth → point cloud → support plane → oriented L/W/H → confidence. Artifacts saved under `outputs/BOX_001/image_001/`.

In [ ]:
import os, json
os.environ['MEASUREX_VISION_OUTPUT_DIR'] = f'outputs/{OBJECT_ID}/image_001'
import torch
print('providers: detector=central_candidate, segmenter=sam2, depth=depth_pro (PRODUCTION_ALLOWED=false)')
print('GPU:', torch.cuda.get_device_name(0))
!python -m measurex_vision.cli image "$img" --provider depth_pro --segmenter sam2 --debug

mj = f'outputs/{OBJECT_ID}/image_001/measurement.json'
print('--- measurement.json ---'); print(open(mj).read()) if os.path.exists(mj) else print('(no measurement.json — see output above)')

## 8. Show / list debug artifacts

In [ ]:
from IPython.display import Image as IPyImage, display
import os, glob
d = f'outputs/{OBJECT_ID}/image_001'
for f in ['image.png', 'mask_overlay.png', 'depth_color.png']:
    p = os.path.join(d, f)
    print(p); display(IPyImage(p)) if os.path.exists(p) else print('  (missing)')
print('all artifacts:', sorted(os.path.basename(x) for x in glob.glob(d + '/*')))

## 9. Run short-video inference (M4 temporal stabilization)

Prints stabilized L/W/H, accepted/rejected frames, per-dimension spread, confidence, runtime.

In [ ]:
import os
os.environ['MEASUREX_VISION_OUTPUT_DIR'] = f'outputs/{OBJECT_ID}/video_001'
!python -m measurex_vision.cli video "$vid" --provider depth_pro --segmenter sam2 --debug
mj = f'outputs/{OBJECT_ID}/video_001/measurement.json'
print('--- video measurement.json ---'); print(open(mj).read()) if os.path.exists(mj) else None

## 10. M5 benchmark vs ground truth

Builds the manifest from your uploaded photo + video and runs the benchmark (per-dimension + CBM error, success / explicit-failure / **false-success** rate, single- vs multi-frame, runtime). `--false-success-tolerance-pct 10` flags any dimension off by >10%.

In [ ]:
import csv, os, shutil
base = f'data/benchmark/objects/{OBJECT_ID}/phoneA'
os.makedirs(base, exist_ok=True)
shutil.copy(img, f'{base}/image_001.jpg')
shutil.copy(vid, f'{base}/video_001.mp4')
with open('data/benchmark/manifest.csv', 'w', newline='') as fh:
    w = csv.writer(fh)
    w.writerow(['sample_id','object_id','category','phone_model','capture_type','file_path',
                'gt_length_mm','gt_width_mm','gt_height_mm','lighting','distance_band','notes'])
    w.writerow([f'{OBJECT_ID}_IMG_001',OBJECT_ID,'medium','phoneA','image',
                f'objects/{OBJECT_ID}/phoneA/image_001.jpg',L,W,H,'warehouse','near','real box'])
    w.writerow([f'{OBJECT_ID}_VID_001',OBJECT_ID,'medium','phoneA','video',
                f'objects/{OBJECT_ID}/phoneA/video_001.mp4',L,W,H,'warehouse','near','real box'])
!python scripts/benchmark.py data/benchmark/manifest.csv --false-success-tolerance-pct 10
print(open('outputs/benchmark/summary.json').read())

## 11. Depth provider comparison

Only Depth Pro is implemented this milestone. Depth Anything V2 Small (the commercial-path candidate) is **deferred** — see `MODELS.md`. When it is implemented, re-run the **same** image/video through both and fill the table in `docs/REAL_GPU_VALIDATION_REPORT.md`.

In [ ]:
print('provider          | PRODUCTION_ALLOWED | status')
print('depth_pro          | false              | used (research/benchmark only)')
print('depth_anything_v2  | pending            | NOT IMPLEMENTED (deferred this milestone)')

## 12. Download results

Record the numbers in `docs/REAL_GPU_VALIDATION_REPORT.md`.

In [ ]:
import shutil
shutil.make_archive('measurex_outputs', 'zip', 'outputs')
from google.colab import files
files.download('measurex_outputs.zip')

## Where results appear

- **Image run:** `outputs/BOX_001/image_001/` → `measurement.json`, `mask_overlay.png`, `depth_color.png`, `depth_raw.npy`, `points.npy`, `cloud_diagnostics.json`.
- **Video run:** `outputs/BOX_001/video_001/`.
- **Benchmark:** `outputs/benchmark/` → `results.csv`, `summary.json`, SVG charts.

**Reading a result:** `success:true` + `dimensions` = predicted L/W/H (mm); `confidence_level` is capped below `high` (uncalibrated). `success:false` + `failure_code` means the scene wasn't measurable — recapture per step 6. Flutter integration starts **only** after you review the report and approve.